Install Libraries

In [1]:
!pip install -q langchain-huggingface huggingface_hub pydantic

Imports & API Token

In [ ]:
import os
import json
from datetime import datetime, timedelta
from typing import Optional, List, Dict, Any, Union
from pydantic import BaseModel, Field

from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# Paste your active Hugging Face Token here
os.environ["HUGGINGFACEHUB_API_TOKEN"] = YOUR_HUGGINGFACE_TOKEN

# Initialize LLM Engine
repo_id = "Qwen/Qwen2.5-Coder-32B-Instruct"
endpoint = HuggingFaceEndpoint(
    repo_id=repo_id,
    task="text-generation",
    max_new_tokens=512,
    temperature=0.1
)
llm = ChatHuggingFace(llm=endpoint)
print("✅ LLM Connected Successfully!")

✅ LLM Connected Successfully!


Required Tools Implementation

In [7]:
# --- 1. Schedule Management Tool ---
# Database simulation for events
EVENTS_DB = []

@tool
def schedule_management_tool(action: str, event_name: str, date: str, time: str) -> str:
    """
    Manages schedule events (create, update, delete) and checks for scheduling conflicts.
    action: 'create', 'update', or 'delete'
    date: YYYY-MM-DD
    time: HH:MM
    """
    global EVENTS_DB
    new_event = {"event_name": event_name, "date": date, "time": time}

    if action.lower() == "create":
        # Check conflicts
        for event in EVENTS_DB:
            if event["date"] == date and event["time"] == time:
                return f"⚠️ Conflict Detected! An event '{event['event_name']}' is already scheduled at {date} {time}."
        EVENTS_DB.append(new_event)
        return f"✅ Event '{event_name}' successfully scheduled for {date} at {time}."

    elif action.lower() == "delete":
        EVENTS_DB = [e for e in EVENTS_DB if not (e["event_name"] == event_name and e["date"] == date)]
        return f"🗑️ Event '{event_name}' on {date} has been deleted."

    elif action.lower() == "update":
        for event in EVENTS_DB:
            if event["event_name"] == event_name:
                event["date"] = date
                event["time"] = time
                return f"🔄 Event '{event_name}' updated to {date} at {time}."
        EVENTS_DB.append(new_event)
        return f"✅ Event '{event_name}' was not found, so it was created for {date} at {time}."

    return "❌ Invalid action specified."

# --- 2. Location Info Tool ---
@tool
def location_info_tool(location_name: str) -> str:
    """
    Returns Country, Timezone, and Current Local Time for a given location.
    """
    locations_db = {
        "cairo": {"country": "Egypt", "timezone": "EEST (UTC+3)", "utc_offset": 3},
        "banha": {"country": "Egypt", "timezone": "EEST (UTC+3)", "utc_offset": 3},
        "london": {"country": "United Kingdom", "timezone": "BST (UTC+1)", "utc_offset": 1},
        "new york": {"country": "United States", "timezone": "EDT (UTC-4)", "utc_offset": -4},
        "tokyo": {"country": "Japan", "timezone": "JST (UTC+9)", "utc_offset": 9}
    }

    loc_key = location_name.lower().strip()
    if loc_key in locations_db:
        info = locations_db[loc_key]
        now_utc = datetime.utcnow()
        local_time = now_utc + timedelta(hours=info["utc_offset"])
        return (f"📍 Location: {location_name.capitalize()}\n"
                f"🌍 Country: {info['country']}\n"
                f"⏰ Time Zone: {info['timezone']}\n"
                f"🕐 Current Local Time: {local_time.strftime('%Y-%m-%d %H:%M:%S')}")
    else:
        return f"❌ Location '{location_name}' is unknown or invalid in the database."

# --- 3. Simple Analytics Tool ---
@tool
def simple_analytics_tool(numbers: List[float]) -> str:
    """
    Computes Average, Maximum, Minimum, and Count for a list of numbers.
    """
    if not numbers or not isinstance(numbers, list):
        return "⚠️ Error: Invalid or empty list of numbers provided."

    try:
        count_val = len(numbers)
        avg_val = sum(numbers) / count_val
        max_val = max(numbers)
        min_val = min(numbers)

        return (f"📊 Analytics Results:\n"
                f"• Count: {count_val}\n"
                f"• Average: {avg_val:.2f}\n"
                f"• Maximum: {max_val}\n"
                f"• Minimum: {min_val}")
    except Exception as e:
        return f"❌ Analytics calculation error: {str(e)}"

print("✅ All Tools Defined Successfully!")

✅ All Tools Defined Successfully!


Agent Architecture & Routing Engine

In [8]:
class AgentExecutor:
    def __init__(self, llm_model):
        self.llm = llm_model
        self.tools = {
            "schedule": schedule_management_tool,
            "location": location_info_tool,
            "analytics": simple_analytics_tool
        }

    def process_request(self, user_query: str):
        print(f"\n==================================================")
        print(f"📥 User Input: '{user_query}'")

        prompt_text = f"""
        You are an AI Agent Decision Engine. Analyze the user query and decide which tool to call and extract structured parameters in JSON format.

        Available Tools:
        1. "schedule": For managing calendar events. Required JSON keys: "action" ('create'/'update'/'delete'), "event_name", "date" (YYYY-MM-DD), "time" (HH:MM).
        2. "location": For fetching country, timezone, local time of a place. Required JSON key: "location_name".
        3. "analytics": For analyzing lists of numbers. Required JSON key: "numbers" (list of floats/ints).

        Return ONLY a JSON object with:
        "tool": name of the tool ("schedule", "location", "analytics"),
        "parameters": dict of parameters.

        User Query: {user_query}
        JSON Response:
        """

        try:
            response = self.llm.invoke(prompt_text)
            content = response.content.strip()

            # Clean JSON formatting
            if "```json" in content:
                content = content.split("```json")[1].split("```")[0].strip()
            elif "```" in content:
                content = content.split("```")[1].split("```")[0].strip()

            decision = json.loads(content)
            tool_name = decision.get("tool")
            params = decision.get("parameters", {})

            print(f"🧠 Detected Tool: [{tool_name}]")
            print(f"🧩 Extracted Parameters: {json.dumps(params, ensure_ascii=False)}")

            # Execute Selected Tool
            if tool_name == "schedule":
                tool_output = schedule_management_tool.invoke(params)
            elif tool_name == "location":
                tool_output = location_info_tool.invoke(params)
            elif tool_name == "analytics":
                tool_output = simple_analytics_tool.invoke(params)
            else:
                tool_output = "❌ Unknown tool requested."

            print(f"⚙️ Tool Output:\n{tool_output}")
            return tool_output

        except Exception as e:
            print(f"❌ Execution Error: {str(e)}")

agent = AgentExecutor(llm)
print("✅ AI Agent Engine Ready!")

✅ AI Agent Engine Ready!


Run Test Cases

In [10]:
# Test Case 1: Schedule Management
agent.process_request("Schedule a team meeting on 2026-08-15 at 10:00")

# Test Case 2: Schedule Conflict Detection
agent.process_request("Schedule a 1-on-1 session on 2026-08-15 at 10:00")

# Test Case 3: Location Info Tool
agent.process_request("What is the current local time and timezone for Cairo?")

# Test Case 4: Simple Analytics Tool
agent.process_request("Calculate the average, max, and min for these exam scores: 85, 90, 78, 92, 88")


📥 User Input: 'Schedule a team meeting on 2026-08-15 at 10:00'
🧠 Detected Tool: [schedule]
🧩 Extracted Parameters: {"action": "create", "event_name": "team meeting", "date": "2026-08-15", "time": "10:00"}
⚙️ Tool Output:
⚠️ Conflict Detected! An event '1-on-1 session' is already scheduled at 2026-08-15 10:00.

📥 User Input: 'Schedule a 1-on-1 session on 2026-08-15 at 10:00'
🧠 Detected Tool: [schedule]
🧩 Extracted Parameters: {"action": "create", "event_name": "1-on-1 session", "date": "2026-08-15", "time": "10:00"}
⚙️ Tool Output:
⚠️ Conflict Detected! An event '1-on-1 session' is already scheduled at 2026-08-15 10:00.

📥 User Input: 'What is the current local time and timezone for Cairo?'
🧠 Detected Tool: [location]
🧩 Extracted Parameters: {"location_name": "Cairo"}
⚙️ Tool Output:
📍 Location: Cairo
🌍 Country: Egypt
⏰ Time Zone: EEST (UTC+3)
🕐 Current Local Time: 2026-08-11 18:22:01

📥 User Input: 'Calculate the average, max, and min for these exam scores: 85, 90, 78, 92, 88'


/tmp/ipykernel_1642/3462070477.py:56: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now_utc = datetime.utcnow()


🧠 Detected Tool: [analytics]
🧩 Extracted Parameters: {"numbers": [85, 90, 78, 92, 88]}
⚙️ Tool Output:
📊 Analytics Results:
• Count: 5
• Average: 86.60
• Maximum: 92.0
• Minimum: 78.0


'📊 Analytics Results:\n• Count: 5\n• Average: 86.60\n• Maximum: 92.0\n• Minimum: 78.0'